# gold_revenue_per_restaurant

**Sources:**
- fact:      `silver.order_items`   (order_item_id, order_id, subtotal, discount_applied)
- bridge:    `silver.orders`        (order_id → restaurant_key which is the CNPJ)
- dimension: `silver.restaurants`   (cnpj → name, city, cuisine_type)

**Joins (two separate aggregation paths, merged at restaurant grain):**
- order-level: `silver.orders` grouped by `restaurant_key` directly — `avg_order_value_brl`
  and `total_orders` are computed here, before any contact with `order_items`, so each
  order's `total_amount` is counted exactly once.
- item-level: `order_items.order_id = orders.order_id` (bridge only, to recover
  `restaurant_key`), then grouped by `restaurant_key` — `total_items_sold`,
  `total_revenue_brl`, `avg_discount_brl`.
- the two restaurant-grain aggregates are joined to each other, then to
  `restaurants.cnpj` (CNPJ is the business key — CLAUDE.md).

> Computing order-level averages *after* joining to `order_items` would repeat each
> order's `total_amount` once per item it contains, inflating `avg_order_value_brl`
> for restaurants with multi-item orders. Keeping the two aggregations separate until
> the final restaurant-grain join avoids that fan-out bias.

**Grain:** one row per restaurant CNPJ  
**Merge key:** `restaurant_cnpj`  
**Lineage:** Silver → Gold (never Bronze directly)

In [ ]:
dbutils.widgets.removeAll()
dbutils.widgets.text("catalog", "ubereats_dev")

In [ ]:
catalog              = dbutils.widgets.get("catalog")
gold_table           = f"{catalog}.gold.revenue_per_restaurant"
silver_order_items   = f"{catalog}.silver.order_items"
silver_orders        = f"{catalog}.silver.orders"
silver_restaurants   = f"{catalog}.silver.restaurants"

print(f"[gold] {gold_table}  ←  {silver_order_items} × {silver_orders} × {silver_restaurants}")

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {gold_table} (
        restaurant_cnpj    STRING NOT NULL,
        restaurant_name    STRING,
        city               STRING,
        cuisine_type       STRING,
        total_orders       LONG,
        total_items_sold   LONG,
        total_revenue_brl  DOUBLE,
        avg_order_value_brl DOUBLE,
        avg_discount_brl   DOUBLE,
        _computed_at       TIMESTAMP NOT NULL
    ) USING DELTA
    CLUSTER BY (restaurant_cnpj)
    TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
""")
print(f"[gold] table {gold_table} ready")

In [ ]:
from pyspark.sql.functions import col, count, sum, avg, countDistinct, current_timestamp

order_items  = spark.table(silver_order_items)
orders       = spark.table(silver_orders)
restaurants  = spark.table(silver_restaurants)

# order-level metrics computed BEFORE any join with order_items — joining first
# would repeat each order's total_amount once per item it contains, inflating
# avg_order_value_brl for restaurants with multi-item orders.
orders_agg = (
    orders
    .groupBy("restaurant_key")
    .agg(
        countDistinct("order_id").alias("total_orders"),
        avg("total_amount").alias("avg_order_value_brl"),
    )
)

# item-level metrics: order_items is the "many" side of the bridge join to
# orders, so this join only attaches restaurant_key — it does not duplicate
# any order_items row, so count/sum/avg over order_items columns stay correct.
items_agg = (
    order_items.alias("i")
    .join(
        orders.select(col("order_id"), col("restaurant_key")).alias("o"),
        col("i.order_id") == col("o.order_id"),
        "inner",
    )
    .groupBy(col("o.restaurant_key"))
    .agg(
        count(col("i.order_item_id")).alias("total_items_sold"),
        sum(col("i.subtotal")).alias("total_revenue_brl"),
        avg(col("i.discount_applied")).alias("avg_discount_brl"),
    )
)

# Merge the two restaurant-grain aggregates, then enrich with dimension silver.restaurants
revenue_df = (
    items_agg
    .join(orders_agg, on="restaurant_key", how="left")
    .join(
        restaurants.select(
            col("cnpj"),
            col("name").alias("restaurant_name"),
            col("city"),
            col("cuisine_type"),
        ),
        items_agg["restaurant_key"] == col("cnpj"),
        "left",
    )
    .select(
        items_agg["restaurant_key"].alias("restaurant_cnpj"),
        col("restaurant_name"),
        col("city"),
        col("cuisine_type"),
        col("total_orders"),
        col("total_items_sold"),
        col("total_revenue_brl"),
        col("avg_order_value_brl"),
        col("avg_discount_brl"),
    )
    .withColumn("_computed_at", current_timestamp())
)

print(f"[gold] {revenue_df.count()} restaurants")

In [ ]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number, desc, col

# silver.restaurants is deduped/merged by uuid, not cnpj (DESIGN_GOLD_DIMENSION_JOIN_INTEGRITY.md).
# The new check=unique rule on cnpj (contracts/restaurants.yml) should keep this from
# happening going forward, but this guard stays as defense-in-depth for any rows that
# landed before the rule existed.
w = Window.partitionBy("restaurant_cnpj").orderBy(desc("_computed_at"))
revenue_df_deduped = (
    revenue_df
    .withColumn("_rn", row_number().over(w))
    .filter(col("_rn") == 1)
    .drop("_rn")
)

revenue_df_deduped.createOrReplaceTempView("gold_revenue_per_restaurant_batch")

spark.sql(f"""
    MERGE INTO {gold_table} AS t
    USING gold_revenue_per_restaurant_batch AS s
    ON t.restaurant_cnpj = s.restaurant_cnpj
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

print(f"[gold] {gold_table} updated")